In [ ]:
import pandas as pd
import numpy as np
import ast
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from collections import Counter

In [ ]:
movies = pd.read_csv('/content/drive/MyDrive/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/drive/MyDrive/tmdb_5000_credits.csv')

In [ ]:
movies = movies.merge(credits, on='title')

In [ ]:
movies = movies[[
    'movie_id',
    'title',
    'overview',
    'genres',
    'keywords',
    'cast',
    'crew',
    'vote_average',
    'vote_count'
]]

In [ ]:
movies.dropna(inplace=True)

In [ ]:
def extract_names(text):
    names = []

    for item in ast.literal_eval(text):
        names.append(item['name'])

    return names

In [ ]:
def extract_cast(text):
    cast_list = []

    counter = 0

    for item in ast.literal_eval(text):

        if counter != 3:
            cast_list.append(item['name'])
            counter += 1
        else:
            break

    return cast_list

In [ ]:
def extract_director(text):

    director = []

    for item in ast.literal_eval(text):

        if item['job'] == 'Director':
            director.append(item['name'])
            break

    return director

In [ ]:
movies['genres'] = movies['genres'].apply(extract_names)

movies['keywords'] = movies['keywords'].apply(extract_names)

movies['cast'] = movies['cast'].apply(extract_cast)

movies['crew'] = movies['crew'].apply(extract_director)

In [ ]:
def collapse_spaces(items):
    return [i.replace(" ", "") for i in items]

In [ ]:
movies['genres'] = movies['genres'].apply(collapse_spaces)

movies['keywords'] = movies['keywords'].apply(collapse_spaces)

movies['cast'] = movies['cast'].apply(collapse_spaces)

movies['crew'] = movies['crew'].apply(collapse_spaces)

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [ ]:
movies['tags'] = (
    movies['overview'] +
    movies['genres'] +
    movies['keywords'] +
    movies['cast'] +
    movies['crew']
)

In [ ]:
new_df = movies[['movie_id', 'title', 'tags']].copy()

In [ ]:
new_df['tags'] = new_df['tags'].apply(
    lambda x: " ".join(x)
)

In [ ]:
new_df['tags'] = new_df['tags'].apply(
    lambda x: x.lower()
)

In [ ]:
from nltk.stem.porter import PorterStemmer

In [ ]:
ps = PorterStemmer()

In [ ]:
def stem(text):

    words = []

    for i in text.split():
        words.append(ps.stem(i))

    return " ".join(words)

In [ ]:
new_df['tags'] = new_df['tags'].apply(stem)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

vectors = tfidf.fit_transform(
    new_df['tags']
).toarray()

In [ ]:
similarity = cosine_similarity(vectors)

In [ ]:
def get_recommendations(movie_title, n=10):

    movie_title = movie_title.lower()

    matching_movies = new_df[
        new_df['title'].str.lower() == movie_title
    ]

    if matching_movies.empty:
        return "Movie not found"

    movie_index = matching_movies.index[0]

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:n+1]

    recommendations = []

    for i in movie_list:
        recommendations.append(
            new_df.iloc[i[0]].title
        )

    return recommendations

In [ ]:
get_recommendations('Avatar')

['Aliens',
 'Falcon Rising',
 'Battle: Los Angeles',
 'Aliens vs Predator: Requiem',
 'Apollo 18',
 'Star Trek Into Darkness',
 'Meet Dave',
 'Predators',
 'Titan A.E.',
 'The Book of Life']

In [ ]:
test_movies = [
    'Avatar',
    'Titanic',
    'Iron Man',
    'Batman Begins',
    'The Dark Knight'
]

for movie in test_movies:

    print("\n")
    print("="*60)
    print(f"Recommendations for: {movie}")
    print("="*60)

    recommendations = get_recommendations(movie)

    for idx, rec in enumerate(recommendations, start=1):
        print(f"{idx}. {rec}")



Recommendations for: Avatar
1. Aliens
2. Falcon Rising
3. Battle: Los Angeles
4. Aliens vs Predator: Requiem
5. Apollo 18
6. Star Trek Into Darkness
7. Meet Dave
8. Predators
9. Titan A.E.
10. The Book of Life


Recommendations for: Titanic
1. Ghost Ship
2. Poseidon
3. Pirates of the Caribbean: On Stranger Tides
4. The Rose
5. Dear Frankie
6. The Black Hole
7. The Bounty
8. 20,000 Leagues Under the Sea
9. Triangle
10. Supernova


Recommendations for: Iron Man
1. Iron Man 2
2. Iron Man 3
3. Avengers: Age of Ultron
4. Ant-Man
5. Captain America: Civil War
6. The Avengers
7. Thor: The Dark World
8. Guardians of the Galaxy
9. The Incredible Hulk
10. Thor


Recommendations for: Batman Begins
1. The Dark Knight
2. The Dark Knight Rises
3. Batman v Superman: Dawn of Justice
4. Batman
5. Batman Returns
6. Batman
7. Batman & Robin
8. Batman Forever
9. Defendor
10. Batman: The Dark Knight Returns, Part 2


Recommendations for: The Dark Knight
1. The Dark Knight Rises
2. Batman Returns
3. Batma

In [ ]:
pickle.dump(
    new_df,
    open('movies.pkl', 'wb')
)

In [ ]:
pickle.dump(
    similarity,
    open('similarity.pkl', 'wb')
)

In [ ]:
from google.colab import files

files.download('movies.pkl')
files.download('similarity.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>